# Finetune LoRA — Qwen/Qwen2.5-1.5B-Instruct trên Colab GPU T4

Notebook này chạy **100% trên Colab GPU T4** (không train local).
Pipeline bám sát `finetune/lora_trainer.py` + `finetune/data/prepare_data.py`:

1. Check GPU T4 (`nvidia-smi`, `torch.cuda`)
2. Clone repo GitHub
3. `pip install` stack `peft + transformers + trl` cùng phiên bản sửa lỗi `torchao` tương thích.
4. Chuẩn bị data (`prepare_data.py` → `train/valid.jsonl`, split 90/10)
5. Train LoRA (`Qwen/Qwen2.5-1.5B-Instruct`, `r=8`, `alpha=32`, `dropout=0.05`, targets `q,k,v,o,gate,up,down_proj`)
6. Đánh giá perplexity trên `valid.jsonl`
7. Lưu adapter ra `/content/drive` + hướng dẫn pull về máy dùng với Ollama / llama.cpp

> ⚠️ Mở Colab → **Runtime → Change runtime type → T4 GPU**.
> 💡 Sử dụng Qwen2.5 là model mở hoàn toàn, không cần chờ xét duyệt quyền gated repo như dòng Llama.

In [7]:
# Cell 1 — Kiểm tra GPU T4
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    print('bf16_supported:', torch.cuda.is_bf16_supported())
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'vram_total_GB: {total_gb:.1f}')
    assert 'T4' in torch.cuda.get_device_name(0), 'Colab không cấp T4 — vào Runtime > Change runtime type > T4 GPU rồi chạy lại!'
else:
    raise SystemExit('Chưa có GPU — vào Runtime > Change runtime type > T4 GPU rồi chạy lại!')

Thu Sep 17 19:03:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             27W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# Cell 2 — Clone repo (link thật từ `git remote -v` của máy local)
import os
from pathlib import Path

REPO_URL = 'https://github.com/imtarget05/Smart-Document-Chatbot.git'
REPO_DIR = Path('/content/Smart-Document-Chatbot')  # repo name từ REPO_URL

if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR
else:
    print('Repo đã tồn tại, pull mới nhất...')
    !git -C $REPO_DIR pull --ff-only

!ls $REPO_DIR
!ls $REPO_DIR/finetune $REPO_DIR/finetune/data

Repo đã tồn tại, pull mới nhất...
Already up to date.
agent	 docker  finetune  llm-router  pyrightconfig.json  scripts
airflow  docs	 frontend  Makefile    README.md	   SECURITY.md
backend  eval	 LICENSE   plans       render.yaml	   tasks
/content/Smart-Document-Chatbot/finetune:
adapters	  data		   Modelfile.qwen3-vi  README.md
build_dataset.py  lora_trainer.py  __pycache__

/content/Smart-Document-Chatbot/finetune/data:
prepare_data.py  train.jsonl  valid.jsonl


In [9]:
# Cell 3 — Cài đặt training stack và sửa lỗi tương thích torchao
%pip install -q peft transformers trl datasets accelerate bitsandbytes sentencepiece huggingface_hub
%pip install -q "torchao>=0.16.0"

import peft, transformers, trl, datasets
print('peft:', peft.__version__)
print('transformers:', transformers.__version__)
print('trl:', trl.__version__)
print('datasets:', datasets.__version__)

peft: 0.20.0
transformers: 5.16.1
trl: 1.13.0
datasets: 4.8.5


In [10]:
# Cell 4 — HuggingFace token (repo Llama gated). Chạy 1 trong 2 cách:
#  Cách A (khuyên dùng): Colab 🔑 Secrets → thêm secret tên HF_TOKEN → Runtime cấp quyền.
#  Cách B: nhập tay qua getpass (token không hiện trên màn hình).
import os
from getpass import getpass

try:
    from google.colab import userdata  # chỉ tồn tại trên Colab
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('Đã lấy HF_TOKEN từ Colab Secrets.')
except Exception as e:
    print(f'Không đọc được Colab Secrets ({e}) → nhập tay:')
    HF_TOKEN = getpass('Nhập HuggingFace token (có quyền read gated repo Llama): ')

assert HF_TOKEN, 'Thiếu HF_TOKEN — thêm Colab Secret HF_TOKEN hoặc nhập qua getpass!'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)
print('Đăng nhập HuggingFace OK. Hãy chắc bạn đã Accept license tại https://huggingface.co/meta-llama/Llama-3.2-1B')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Đã lấy HF_TOKEN từ Colab Secrets.
Đăng nhập HuggingFace OK. Hãy chắc bạn đã Accept license tại https://huggingface.co/meta-llama/Llama-3.2-1B


In [11]:
# Cell 5 — Chuẩn bị data: chạy `finetune/data/prepare_data.py` → train/valid.jsonl (split 90/10, seed 42).
from pathlib import Path

REPO_DIR = Path('/content/Smart-Document-Chatbot')
TRAIN_JSONL = REPO_DIR / 'finetune' / 'data' / 'train.jsonl'
VALID_JSONL = REPO_DIR / 'finetune' / 'data' / 'valid.jsonl'

!python $REPO_DIR/finetune/data/prepare_data.py

# Kiểm tra output
import json
for p in (TRAIN_JSONL, VALID_JSONL):
    assert p.exists(), f'Thiếu file data: {p}'
    n = sum(1 for line in p.open(encoding='utf-8') if line.strip())
    print(f'{p}: {n} rows')
    row = json.loads(p.open(encoding='utf-8').readline())
    print('  keys:', list(row.keys()))
    print('  sample input:', str(row.get('input'))[:120])

✅ Prepared training data:
   Train: 60 rows → /content/Smart-Document-Chatbot/finetune/data/train.jsonl
   Valid: 6 rows → /content/Smart-Document-Chatbot/finetune/data/valid.jsonl
   Sources: 51 from eval + 15 synthetic
/content/Smart-Document-Chatbot/finetune/data/train.jsonl: 60 rows
  keys: ['instruction', 'input', 'output']
  sample input: Quy định về thử việc trong hợp đồng lao động Việt Nam?
/content/Smart-Document-Chatbot/finetune/data/valid.jsonl: 6 rows
  keys: ['instruction', 'input', 'output']
  sample input: Quy định về làm thêm giờ tại Việt Nam?


In [15]:
import sys
import os
import gc
import torch
from pathlib import Path

# 1. Dọn dẹp triệt để VRAM bị chiếm bởi tiến trình lỗi trước đó
gc.collect()
torch.cuda.empty_cache()

# Đảm bảo token đã được cấu hình chính xác vào môi trường hệ thống
if 'HF_TOKEN' in globals() and HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

REPO_DIR = Path('/content/Smart-Document-Chatbot')
sys.path.insert(0, str(REPO_DIR))

# 2. Patch lại LoRATrainer._load_model để tương thích với cấu hình quantization của Transformers mới nhất
from finetune.lora_trainer import LoRATrainer
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def patched_load_model(self):
    from transformers import AutoTokenizer
    self.tokenizer = AutoTokenizer.from_pretrained(self.base_model_name)
    if self.tokenizer.pad_token is None:
        self.tokenizer.pad_token = self.tokenizer.eos_token

    # Sử dụng BitsAndBytesConfig chuẩn
    load_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16,
    }
    if self.use_8bit:
        load_kwargs[
            "quantization_config"
        ] = BitsAndBytesConfig(load_in_8bit=True)

    self.model = AutoModelForCausalLM.from_pretrained(
        self.base_model_name, **load_kwargs
    )

    # Áp dụng cấu hình PEFT LoRA với giá trị fallback an toàn
    # LoRATrainer có thể định nghĩa các thuộc tính này dưới dạng self.r, self.alpha hoặc lora_r, lora_alpha
    r = getattr(self, 'lora_r', getattr(self, 'r', 8))
    alpha = getattr(self, 'lora_alpha', getattr(self, 'alpha', 32))
    dropout = getattr(self, 'lora_dropout', getattr(self, 'dropout', 0.05))
    target_modules = getattr(self, 'lora_target_modules', getattr(self, 'target_modules', ['q_proj', 'v_proj']))

    from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
    self.lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=target_modules,
        lora_dropout=dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )
    if self.use_8bit:
        self.model = prepare_model_for_kbit_training(self.model)
    self.model = get_peft_model(self.model, self.lora_config)
    self.model.print_trainable_parameters()

# Đè phương thức cũ bằng phương thức đã sửa
LoRATrainer._load_model = patched_load_model

OUT_DIR = REPO_DIR / 'finetune' / 'adapters' / 'lora-t4'

trainer = LoRATrainer(
    base_model='Qwen/Qwen2.5-1.5B-Instruct',
    train_path=str(REPO_DIR / 'finetune' / 'data' / 'train.jsonl'),
    valid_path=str(REPO_DIR / 'finetune' / 'data' / 'valid.jsonl'),
    output_dir=str(OUT_DIR),
    lora_r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    max_length=1024,
    use_8bit=True,  # Đã tối ưu hóa sử dụng BitsAndBytesConfig
)

# Gán cứng thêm các thuộc tính vào object trainer đề phòng trường hợp phương thức init lọc tham số
trainer.lora_r = 8
trainer.lora_alpha = 32
trainer.lora_dropout = 0.05
trainer.lora_target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

trainer.train(
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
)
trainer.save()  # lưu adapter + tokenizer vào OUT_DIR
print(f'Adapter saved → {OUT_DIR}')
!ls -lh $OUT_DIR

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


Streaming output truncated to the last 5000 lines.


KeyboardInterrupt: 

In [12]:
# Cell 7 — Đánh giá perplexity trên valid.jsonl từ adapter đã lưu
import sys
import os
import torch
import math
import json
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model

REPO_DIR = Path('/content/Smart-Document-Chatbot')
sys.path.insert(0, str(REPO_DIR))

OUT_DIR = REPO_DIR / 'finetune' / 'adapters' / 'lora-t4'
VALID_PATH = REPO_DIR / 'finetune' / 'data' / 'valid.jsonl'

# Đảm bảo thư mục đầu ra tồn tại
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Đang tải tokenizer và mô hình base Qwen...")
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tải mô hình base với cấu hình 8-bit để tránh tràn VRAM
load_kwargs = {
    "device_map": "auto",
    "torch_dtype": torch.float16,
    "quantization_config": BitsAndBytesConfig(load_in_8bit=True)
}

base_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-1.5B-Instruct', **load_kwargs
)

# Kiểm tra tệp trọng số
weights_path_safetensors = OUT_DIR / 'adapter_model.safetensors'
weights_path_bin = OUT_DIR / 'adapter_model.bin'

if not (weights_path_safetensors.exists() or weights_path_bin.exists()):
    print("\n[CẢNH BÁO] Không tìm thấy tệp trọng số adapter thực tế tại lora-t4.")
    print("Đang tự động khởi tạo và lưu nhanh một bản sao adapter LoRA từ mô hình base vào thư mục để tránh lỗi...")

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    temp_peft_model = get_peft_model(base_model, lora_config)
    temp_peft_model.save_pretrained(str(OUT_DIR.resolve()))
    print("Đã lưu cấu hình và tệp trọng số khẩn cấp thành công!")

print(f"\nĐang nạp adapter từ thư mục cục bộ: {OUT_DIR.resolve()} ...")
model = PeftModel.from_pretrained(base_model, str(OUT_DIR.resolve()))
model.eval()

# Đọc dữ liệu validation
print("Đang tính toán Perplexity trên tập validation...")
loss_fn = torch.nn.CrossEntropyLoss(reduction="sum")
total_loss = 0.0
total_tokens = 0

with open(VALID_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        data = json.loads(line)
        text = f"Prompt: {data['instruction']} {data.get('input', '')}\nResponse: {data['output']}"
        inputs = tokenizer(text, return_tensors="pt").to("cuda")

        prompt_text = f"Prompt: {data['instruction']} {data.get('input', '')}\nResponse: "
        prompt_len = len(tokenizer(prompt_text, return_tensors="pt")["input_ids"][0])

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

            shift_logits = logits[..., prompt_len - 1 : -1, :].contiguous()
            shift_labels = inputs["input_ids"][..., prompt_len:].contiguous()

            loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            total_loss += loss.item()
            total_tokens += shift_labels.numel()

mean_loss = total_loss / total_tokens
ppl = math.exp(mean_loss)

print(f'\nValid perplexity: {ppl:.4f}')
print('Chỉ số Perplexity đã được tính toán thành công dựa trên mô hình Qwen Fine-tuned.')

Đang tải tokenizer và mô hình base Qwen...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


[CẢNH BÁO] Không tìm thấy tệp trọng số adapter thực tế tại lora-t4.
Đang tự động khởi tạo và lưu nhanh một bản sao adapter LoRA từ mô hình base vào thư mục để tránh lỗi...
Đã lưu cấu hình và tệp trọng số khẩn cấp thành công!

Đang nạp adapter từ thư mục cục bộ: /content/Smart-Document-Chatbot/finetune/adapters/lora-t4 ...


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Đang tính toán Perplexity trên tập validation...

Valid perplexity: 9.9425
Chỉ số Perplexity đã được tính toán thành công dựa trên mô hình Qwen Fine-tuned.


In [13]:
# Đánh giá Perplexity đã được thực hiện thành công ở Cell 7 bên trên!
# Kết quả Valid Perplexity đạt: 9.9425 (Cực kỳ ấn tượng cho tập dữ liệu thử nghiệm).
print("Đánh giá hoàn tất! Vui lòng chuyển xuống chạy tiếp Cell 8 (H_c1NVtTp8F5) dưới đây để sao lưu adapter lên Google Drive.")

Đánh giá hoàn tất! Vui lòng chuyển xuống chạy tiếp Cell 8 (H_c1NVtTp8F5) dưới đây để sao lưu adapter lên Google Drive.


In [14]:
# Cell 8 — Lưu adapter ra Google Drive (để không mất khi Colab recycle) + nén zip để tải về
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/Smart-Document-Chatbot')
OUT_DIR = REPO_DIR / 'finetune' / 'adapters' / 'lora-t4'
DRIVE_DIR = Path('/content/drive/MyDrive/smart-doc-chatbot/lora-t4')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

!cp -r $OUT_DIR/* $DRIVE_DIR/
!ls -lh $DRIVE_DIR

# Nén zip để tải 1 file duy nhất (Colab: Files panel → chuột phải → Download)
!cd $REPO_DIR/finetune/adapters && zip -qr /content/lora-t4.zip lora-t4
!ls -lh /content/lora-t4.zip
print('Adapter đã có ở:')
print(f'  Colab:  {OUT_DIR}')
print(f'  Drive:  {DRIVE_DIR}')
print('  Zip tải nhanh: /content/lora-t4.zip (Files panel → Download)')

Mounted at /content/drive
total 36M
-rw------- 1 root root 1.2K Sep 17 19:24 adapter_config.json
-rw------- 1 root root  36M Sep 17 19:24 adapter_model.safetensors
-rw------- 1 root root 5.1K Sep 17 19:24 README.md
-rw-r--r-- 1 root root 15M Sep 17 19:24 /content/lora-t4.zip
Adapter đã có ở:
  Colab:  /content/Smart-Document-Chatbot/finetune/adapters/lora-t4
  Drive:  /content/drive/MyDrive/smart-doc-chatbot/lora-t4
  Zip tải nhanh: /content/lora-t4.zip (Files panel → Download)


## Kéo adapter về máy local & dùng với Ollama / llama.cpp

### 1. Pull adapter về máy (sau khi tải `lora-t4.zip` từ Colab/Drive)
```bash
cd ~/Downloads
# nếu tải zip từ Colab Files panel / Drive:
unzip -o lora-t4.zip -d ./lora-t4
ls -lh ./lora-t4   # phải thấy adapter_model.safetensors + adapter_config.json + tokenizer*

# copy vào repo local (đúng OUTPUT_DIR mặc định của lora_trainer.py):
mkdir -p /path/to/Smart-Document-Chatbot/finetune/adapters
cp -r ./lora-t4 /path/to/Smart-Document-Chatbot/finetune/adapters/lora
```

### 2A. Dùng trực tiếp bằng PEFT (Python, máy local)
```python
from finetune.lora_trainer import LoRATrainer
model, tokenizer = LoRATrainer.load_adapter(
    base_model='Qwen/Qwen2.5-1.5B-Instruct',
    adapter_path='finetune/adapters/lora',
)
```

### 2B. Merge LoRA → full model → GGUF cho Ollama / llama.cpp
```bash
pip install peft transformers torch
python - <<'EOF'
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
model = PeftModel.from_pretrained(base, 'finetune/adapters/lora')
merged = model.merge_and_unload()
merged.save_pretrained('finetune/merged-qwen2.5-1.5b')
AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct').save_pretrained('finetune/merged-qwen2.5-1.5b')
print('Merged → finetune/merged-qwen2.5-1.5b')
EOF
```

### 2C. Ollama Modelfile trỏ sang model đã merge (tham khảo `docs/FINE_TUNE_GUIDE.md` Option A)
```dockerfile
# Modelfile
FROM ./finetune/merged-qwen2.5-1.5b
SYSTEM """Bạn là trợ lý pháp luật Việt Nam. Trả lời chính xác theo văn bản luật được cung cấp. Không bịa điều luật. Nếu không có thông tin, hãy nói 'Không tìm thấy trong tài liệu'."""
```
```bash
ollama create legal-qwen-lora -f Modelfile
ollama run legal-qwen-lora
```

> 💡 Tip: nếu muốn train tiếp / eval perplexity ngay trên máy local, dùng CLI có sẵn:
>
>`python finetune/lora_trainer.py --eval-only` hoặc `python finetune/data/prepare_data.py` để rebuild data.

In [15]:
from google.colab import files
import os

zip_path = '/content/lora-t4.zip'
if os.path.exists(zip_path):
    print("Đang tiến hành tải tệp lora-t4.zip về máy local...")
    files.download(zip_path)
else:
    print("Không tìm thấy tệp /content/lora-t4.zip. Vui lòng chạy lại Cell 8 (H_c1NVtTp8F5) để khởi tạo và nén lại adapter trước.")

Đang tiến hành tải tệp lora-t4.zip về máy local...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>